In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
from src.feature_engineering import create_aggregate_features,create_encoder,create_encoding_pipeline,get_categorical_features,handle_missing_values,standardize_features,apply_woe_encoding
from sklearn.preprocessing import LabelEncoder

: 

In [ ]:
df = pd.read_pickle("../data/processed/processed_data.pkl")
df

,TransactionId,BatchId,AccountId,SubscriptionId,CustomerId,CurrencyCode,CountryCode,ProviderId,ProductId,ProductCategory,...,Amount,Value,TransactionStartTime,PricingStrategy,FraudResult,TransactionHour,TransactionDay,TransactionMonth,TransactionYear,TransactionDayOfWeek
0,TransactionId_76871,BatchId_36123,AccountId_3957,SubscriptionId_887,CustomerId_4406,UGX,256,ProviderId_6,ProductId_10,airtime,...,1000.0,1000,2018-11-15 02:18:49+00:00,2,0,2,15,11,2018,3
1,TransactionId_73770,BatchId_15642,AccountId_4841,SubscriptionId_3829,CustomerId_4406,UGX,256,ProviderId_4,ProductId_6,financial_services,...,-20.0,20,2018-11-15 02:19:08+00:00,2,0,2,15,11,2018,3
2,TransactionId_26203,BatchId_53941,AccountId_4229,SubscriptionId_222,CustomerId_4683,UGX,256,ProviderId_6,ProductId_1,airtime,...,500.0,500,2018-11-15 02:44:21+00:00,2,0,2,15,11,2018,3
3,TransactionId_380,BatchId_102363,AccountId_648,SubscriptionId_2185,CustomerId_988,UGX,256,ProviderId_1,ProductId_21,utility_bill,...,20000.0,21800,2018-11-15 03:32:55+00:00,2,0,3,15,11,2018,3
4,TransactionId_28195,BatchId_38780,AccountId_4841,SubscriptionId_3829,CustomerId_988,UGX,256,ProviderId_4,ProductId_6,financial_services,...,-644.0,644,2018-11-15 03:34:21+00:00,2,0,3,15,11,2018,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95657,TransactionId_89881,BatchId_96668,AccountId_4841,SubscriptionId_3829,CustomerId_3078,UGX,256,ProviderId_4,ProductId_6,financial_services,...,-1000.0,1000,2019-02-13 09:54:09+00:00,2,0,9,13,2,2019,2
95658,TransactionId_91597,BatchId_3503,AccountId_3439,SubscriptionId_2643,CustomerId_3874,UGX,256,ProviderId_6,ProductId_10,airtime,...,1000.0,1000,2019-02-13 09:54:25+00:00,2,0,9,13,2,2019,2
95659,TransactionId_82501,BatchId_118602,AccountId_4841,SubscriptionId_3829,CustomerId_3874,UGX,256,ProviderId_4,ProductId_6,financial_services,...,-20.0,20,2019-02-13 09:54:35+00:00,2,0,9,13,2,2019,2
95660,TransactionId_136354,BatchId_70924,AccountId_1346,SubscriptionId_652,CustomerId_1709,UGX,256,ProviderId_6,ProductId_19,tv,...,3000.0,3000,2019-02-13 10:01:10+00:00,2,0,10,13,2,2019,2


# Aggregate Feature Engineering

In [ ]:
customer_features = create_aggregate_features(df)

customer_features.head()

,CustomerId,TotalTransactionValue,AverageTransactionValue,TransactionCount,StdTransactionValue,MaxTransactionValue,MinTransactionValue
0,CustomerId_1,10000,10000.000000,1,NaN,10000,10000
1,CustomerId_10,10000,10000.000000,1,NaN,10000,10000
2,CustomerId_1001,30400,6080.000000,5,4100.243895,10000,200
3,CustomerId_1002,4775,434.090909,11,518.805446,1500,25
4,CustomerId_1003,32000,5333.333333,6,3945.461528,10000,1000


A missing value in the Standard Deviation of Transaction Amounts feature usually means that the customer has only one transaction.

In [ ]:
df = df.merge(
    customer_features,
    on="CustomerId",
    how="left"
)
df

,TransactionId,BatchId,AccountId,SubscriptionId,CustomerId,CurrencyCode,CountryCode,ProviderId,ProductId,ProductCategory,...,TransactionDay,TransactionMonth,TransactionYear,TransactionDayOfWeek,TotalTransactionValue,AverageTransactionValue,TransactionCount,StdTransactionValue,MaxTransactionValue,MinTransactionValue
0,TransactionId_76871,BatchId_36123,AccountId_3957,SubscriptionId_887,CustomerId_4406,UGX,256,ProviderId_6,ProductId_10,airtime,...,15,11,2018,3,203847,1713.000000,119,2675.218372,20000,10
1,TransactionId_73770,BatchId_15642,AccountId_4841,SubscriptionId_3829,CustomerId_4406,UGX,256,ProviderId_4,ProductId_6,financial_services,...,15,11,2018,3,203847,1713.000000,119,2675.218372,20000,10
2,TransactionId_26203,BatchId_53941,AccountId_4229,SubscriptionId_222,CustomerId_4683,UGX,256,ProviderId_6,ProductId_1,airtime,...,15,11,2018,3,1000,500.000000,2,0.000000,500,500
3,TransactionId_380,BatchId_102363,AccountId_648,SubscriptionId_2185,CustomerId_988,UGX,256,ProviderId_1,ProductId_21,utility_bill,...,15,11,2018,3,286623,7542.710526,38,17691.401706,106300,10
4,TransactionId_28195,BatchId_38780,AccountId_4841,SubscriptionId_3829,CustomerId_988,UGX,256,ProviderId_4,ProductId_6,financial_services,...,15,11,2018,3,286623,7542.710526,38,17691.401706,106300,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95657,TransactionId_89881,BatchId_96668,AccountId_4841,SubscriptionId_3829,CustomerId_3078,UGX,256,ProviderId_4,ProductId_6,financial_services,...,13,2,2019,2,2541260,4435.008726,573,22522.192716,235000,10
95658,TransactionId_91597,BatchId_3503,AccountId_3439,SubscriptionId_2643,CustomerId_3874,UGX,256,ProviderId_6,ProductId_10,airtime,...,13,2,2019,2,63117,1467.837209,43,2338.652623,11200,20
95659,TransactionId_82501,BatchId_118602,AccountId_4841,SubscriptionId_3829,CustomerId_3874,UGX,256,ProviderId_4,ProductId_6,financial_services,...,13,2,2019,2,63117,1467.837209,43,2338.652623,11200,20
95660,TransactionId_136354,BatchId_70924,AccountId_1346,SubscriptionId_652,CustomerId_1709,UGX,256,ProviderId_6,ProductId_19,tv,...,13,2,2019,2,998873,1906.246183,524,3053.135486,23000,4


# Categorical Variable Encoding

In [ ]:
encoder = create_encoding_pipeline()

encoded_data = encoder.fit_transform(df)

In [ ]:
feature_names = (
    encoder.get_feature_names_out()
)

processed_df = pd.DataFrame(
    encoded_data,
    columns=feature_names,
    index=df.index
)

# Missing Value Handling

In [ ]:
customer_features = handle_missing_values(
    customer_features
)

The original dataset contained no missing values. However, missing values were introduced during feature engineering when calculating the `StdTransactionValue` feature for customers with only a single transaction. Since transaction variability cannot be computed from a single observation, these missing values were replaced with 0, indicating no observed variation in transaction value. This approach preserved all customer records while maintaining the business meaning of the feature.

In [2]:
numerical_cols = [
    "Value",
    "TotalTransactionValue",
    "AverageTransactionValue",
    "TransactionCount",
    "StdTransactionValue",
    "MaxTransactionValue",
    "MinTransactionValue"
]

df = standardize_features(
    df,
    numerical_cols
)

NameError: name 'standardize_features' is not defined